# Model 2A — Gaussian Beam Propagation in Free Space

This notebook models the propagation of an ideal fundamental Gaussian beam (TEM$_{00}$) in free space.

We will calculate and visualize:

- Beam waist $w_0$
- Rayleigh range $z_R$
- Beam radius $w(z)$
- Wavefront radius of curvature $R(z)$
- Gouy phase $\zeta(z)$
- Far-field divergence angle $\theta$
- Peak intensity $I_0(z)$
- Transverse Gaussian intensity profile
- Beam propagation through free space

This is the free-space model only. ABCD matrices and optical cavities will be introduced in later models.

## 1. Physical picture

A real laser beam has a finite transverse size. For an ideal Gaussian beam, the transverse intensity is

$$
I(r,z)=I_0(z)\exp\left[-\frac{2r^2}{w^2(z)}\right].
$$

The beam has a minimum radius called the **beam waist** $w_0$.

As the beam propagates away from the waist, it expands. The characteristic distance over which this expansion occurs is the **Rayleigh range** $z_R$.

The beam radius is

$$
w(z)=w_0\sqrt{1+\left(\frac{z}{z_R}\right)^2}.
$$

The Rayleigh range is

$$
z_R=\frac{\pi w_0^2}{\lambda}.
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use standard plotting defaults
    

## 2. Define the beam parameters

We will use a 1064 nm laser wavelength, which is relevant to the LIGO optical system.

You can change these values and rerun the notebook.

In [ ]:
# Physical constants
c = 299792458.0          # Speed of light (m/s)

# Beam parameters
wavelength = 1064e-9     # Wavelength (m) = 1064 nm
w0 = 1.0e-3              # Beam waist radius (m) = 1 mm
P = 1.0                  # Optical power (W)

# Propagation range
z_min = -5.0             # Minimum z (m)
z_max = 5.0              # Maximum z (m)
N = 2000                 # Number of points

z = np.linspace(z_min, z_max, N)

print(f"Wavelength = {wavelength*1e9:.1f} nm")
print(f"Beam waist w0 = {w0*1e3:.3f} mm")
print(f"Optical power = {P:.3f} W")

## 3. Rayleigh range

The Rayleigh range is

$$
z_R=\frac{\pi w_0^2}{\lambda}.
$$

At $z=z_R$,

$$
w(z_R)=\sqrt{2}w_0.
$$

So the beam area has doubled relative to the waist area.

In [ ]:
# Rayleigh range
z_R = np.pi * w0**2 / wavelength

print(f"Rayleigh range z_R = {z_R:.4f} m")
print(f"Rayleigh range z_R = {z_R*1000:.2f} mm")

# Sanity check
print("\nAt z = z_R, the beam radius should be sqrt(2) * w0.")
    

## 4. Beam radius during propagation

The Gaussian beam radius is

$$
w(z)=w_0\sqrt{1+\left(\frac{z}{z_R}\right)^2}.
$$

At the waist ($z=0$),

$$
w(0)=w_0.
$$

The beam is symmetric around the waist.

In [ ]:
def beam_radius(z, w0, z_R):
    """Gaussian beam radius w(z)."""
    return w0 * np.sqrt(1 + (z / z_R)**2)


w = beam_radius(z, w0, z_R)

print(f"Minimum beam radius = {np.min(w)*1e3:.4f} mm")
print(f"Beam radius at z = 0 = {beam_radius(0, w0, z_R)*1e3:.4f} mm")
print(f"Beam radius at z = z_R = {beam_radius(z_R, w0, z_R)*1e3:.4f} mm")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(z, w * 1e3)
plt.axvline(0, linestyle='--', label='Beam waist')
plt.axvline(z_R, linestyle=':', label='$+z_R$')
plt.axvline(-z_R, linestyle=':', label='$-z_R$')

plt.xlabel('z (m)')
plt.ylabel('Beam radius w(z) (mm)')
plt.title('Gaussian Beam Radius During Free-Space Propagation')
plt.grid(True)
plt.legend()
plt.show()

### What should you notice?

The beam reaches its minimum radius at $z=0$.

Moving away from the waist in either direction increases the beam radius.

The characteristic scale of this expansion is the Rayleigh range $z_R$.

## 5. Far-field divergence

For an ideal Gaussian beam, the far-field divergence half-angle is

$$
\theta=\frac{\lambda}{\pi w_0}.
$$

This gives the approximate beam expansion at distances much larger than the Rayleigh range:

$$
w(z)\approx \theta |z|.
$$


In [ ]:
theta = wavelength / (np.pi * w0)

print(f"Divergence half-angle = {theta:.6e} rad")
print(f"Divergence half-angle = {theta*1e3:.4f} mrad")
print(f"Full divergence angle ≈ {2*theta*1e3:.4f} mrad")

## 6. Wavefront radius of curvature

The radius of curvature of the Gaussian beam wavefront is

$$
R(z)=z\left[1+\left(\frac{z_R}{z}\right)^2\right].
$$

At the beam waist,

$$
R(0)\rightarrow\infty.
$$

This means that the wavefront is approximately planar at the waist.

In [ ]:
def wavefront_radius(z, z_R):
    """Wavefront radius of curvature R(z)."""
        z = np.asarray(z, dtype=float)
    R = np.empty_like(z)
        nonzero = z != 0
    R[nonzero] = z[nonzero] * (1 + (z_R / z[nonzero])**2)
    R[~nonzero] = np.inf
    return R


R = wavefront_radius(z, z_R)

# Plot only a region where the numerical scale is useful
mask = np.abs(z) > 0.05

plt.figure(figsize=(9, 5))
plt.plot(z[mask], R[mask])
plt.axhline(0, linestyle='--')
plt.axvline(0, linestyle='--', label='Beam waist')
plt.xlabel('z (m)')
plt.ylabel('Wavefront radius R(z) (m)')
plt.title('Gaussian Beam Wavefront Radius of Curvature')
plt.grid(True)
plt.legend()
plt.show()

## 7. Gouy phase

A Gaussian beam acquires an additional phase during propagation called the **Gouy phase**:

$$
\zeta(z)=\tan^{-1}\left(\frac{z}{z_R}\right).
$$

At the waist:

$$
\zeta(0)=0.
$$

Far from the waist:

$$
\zeta(+\infty)=+\frac{\pi}{2},
    $$

$$
\zeta(-\infty)=-\frac{\pi}{2}.
$$

Therefore, the total Gouy phase change through the focus is $\pi$.

In [ ]:
gouy_phase = np.arctan(z / z_R)

plt.figure(figsize=(9, 5))
plt.plot(z, gouy_phase)
plt.axhline(0, linestyle='--')
plt.axvline(0, linestyle='--', label='Beam waist')
plt.xlabel('z (m)')
plt.ylabel('Gouy phase ζ(z) (rad)')
plt.title('Gaussian Beam Gouy Phase')
plt.grid(True)
plt.legend()
plt.show()

print(f"Gouy phase at z = 0: {np.arctan(0):.3f} rad")
print(f"Gouy phase at z = +∞ approaches: {np.pi/2:.3f} rad")
print(f"Gouy phase at z = -∞ approaches: {-np.pi/2:.3f} rad")

## 8. Peak intensity

For a Gaussian beam with total optical power $P$,

$$
I(r,z)=I_0(z)\exp\left[-\frac{2r^2}{w^2(z)}\right].
$$

The peak intensity on the beam axis is

$$
I_0(z)=\frac{2P}{\pi w^2(z)}.
$$

Therefore, as the beam expands, its peak intensity decreases.

In [ ]:
def peak_intensity(w, P):
    """On-axis peak intensity of a Gaussian beam."""
    return 2 * P / (np.pi * w**2)


I0 = peak_intensity(w, P)

plt.figure(figsize=(9, 5))
plt.plot(z, I0 / 1e6)
plt.axvline(0, linestyle='--', label='Beam waist')
plt.xlabel('z (m)')
plt.ylabel('Peak intensity (MW/m²)')
plt.title('Peak Gaussian-Beam Intensity During Propagation')
plt.grid(True)
plt.legend()
plt.show()

print(f"Peak intensity at waist = {I0[N//2]/1e6:.4f} MW/m²")

## 9. Transverse intensity profile

At a particular position $z$, the radial intensity distribution is

$$
I(r,z)=I_0(z)\exp\left[-\frac{2r^2}{w^2(z)}\right].
$$

The radius $w(z)$ is the point where the intensity has dropped to

$$
I(w,z)=I_0(z)e^{-2}.
$$

Thus $w$ is the **1/e² intensity radius**.

In [ ]:
def gaussian_intensity(r, z_value, w0, z_R, P):
    """Gaussian beam intensity I(r,z)."""
    w_z = beam_radius(z_value, w0, z_R)
    I_peak = 2 * P / (np.pi * w_z**2)
        return I_peak * np.exp(-2 * r**2 / w_z**2)


# Choose several propagation positions
z_positions = [0, z_R, 2*z_R]

# Radial coordinate
r = np.linspace(-4*w0, 4*w0, 1000)

plt.figure(figsize=(9, 5))

for z_value in z_positions:
    I = gaussian_intensity(r, z_value, w0, z_R, P)
    plt.plot(r*1e3, I/1e6, label=f'z = {z_value:.2f} m')

plt.xlabel('Transverse position r (mm)')
plt.ylabel('Intensity (MW/m²)')
plt.title('Gaussian Transverse Intensity Profiles')
plt.grid(True)
plt.legend()
plt.show()

## 10. Two-dimensional Gaussian intensity distribution

A useful visualization is the transverse intensity pattern at the beam waist.

Because a fundamental Gaussian beam is cylindrically symmetric, the intensity depends only on the radial distance from the beam axis.

In [ ]:
# 2D transverse coordinate grid
extent = 3*w0
points = 500

x = np.linspace(-extent, extent, points)
y = np.linspace(-extent, extent, points)
    X, Y = np.meshgrid(x, y)
    rho = np.sqrt(X**2 + Y**2)


# Intensity at the waist
I_2D = gaussian_intensity(rho, 0, w0, z_R, P)

plt.figure(figsize=(7, 6))
plt.imshow(
    I_2D / np.max(I_2D),
    extent=[-extent*1e3, extent*1e3, -extent*1e3, extent*1e3],
    origin='lower',
    aspect='equal'
)
plt.xlabel('x (mm)')
plt.ylabel('y (mm)')
plt.title('Normalized Gaussian Beam Intensity at the Waist')
plt.colorbar(label='Normalized intensity')
plt.show()

## 11. Beam propagation visualization

We can visualize the beam envelope using $\pm w(z)$.

The region between these curves represents the approximate transverse extent of the Gaussian beam.

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(z, w*1e3, label='+w(z)')
plt.plot(z, -w*1e3, label='-w(z)')
plt.axvline(0, linestyle='--', label='Beam waist')

plt.fill_between(z, -w*1e3, w*1e3, alpha=0.15)

plt.xlabel('Propagation distance z (m)')
plt.ylabel('Beam radius (mm)')
plt.title('Gaussian Beam Propagation in Free Space')
plt.grid(True)
plt.legend()
plt.show()

## 12. Important values at selected positions

Let's inspect the beam at the waist, one Rayleigh range away, and two Rayleigh ranges away.

In [ ]:
selected_z = np.array([0, z_R, 2*z_R])

print("Position-dependent Gaussian beam parameters")
print("-" * 70)
print(f"{'z (m)':>12} {'w(z) (mm)':>15} {'R(z) (m)':>15} {'Gouy (rad)':>15}")
print("-" * 70)

for z_value in selected_z:
    w_value = beam_radius(z_value, w0, z_R)
    R_value = wavefront_radius(z_value, z_R)
    gouy_value = np.arctan(z_value/z_R)

    if np.isinf(R_value):
        R_string = "infinity"
    else:
        R_string = f"{R_value:.4f}"

    print(f"{z_value:12.4f} {w_value*1e3:15.4f} {R_string:>15} {gouy_value:15.4f}")

## 13. Important sanity checks

These checks help verify that the numerical implementation agrees with the analytical equations.

In [ ]:
# Check 1: Beam radius at waist
assert np.isclose(beam_radius(0, w0, z_R), w0)

# Check 2: Beam radius at Rayleigh range
assert np.isclose(beam_radius(z_R, w0, z_R), np.sqrt(2)*w0)

# Check 3: Symmetry around the waist
assert np.isclose(beam_radius(z_R, w0, z_R), beam_radius(-z_R, w0, z_R))

# Check 4: Gouy phase at waist
assert np.isclose(np.arctan(0), 0)

# Check 5: Divergence relation
assert np.isclose(theta, wavelength/(np.pi*w0))

print("All sanity checks passed.")

## 14. Summary of the Gaussian beam

For an ideal Gaussian beam:

### Rayleigh range
$$
z_R=\frac{\pi w_0^2}{\lambda}
$$

### Beam radius
$$
w(z)=w_0\sqrt{1+\left(\frac{z}{z_R}\right)^2}
$$

### Wavefront curvature
$$
R(z)=z\left[1+\left(\frac{z_R}{z}\right)^2\right]
$$

### Gouy phase
$$
\zeta(z)=\tan^{-1}\left(\frac{z}{z_R}\right)
$$

### Divergence
$$
\theta=\frac{\lambda}{\pi w_0}
$$

### Peak intensity
$$
I_0(z)=\frac{2P}{\pi w^2(z)}
$$

The central physical relationship to remember is:

$$
\boxed{w_0\downarrow\;\Rightarrow\;z_R\downarrow\;\Rightarrow\;\theta\uparrow\;\Rightarrow\;I_0\uparrow}
    $$

This focusing tradeoff will become important when we put a nonlinear crystal inside the cavity.

# What comes next?

This notebook deliberately stops at free-space propagation.

The next model will be **Gaussian Beam + Simple Optical Cavity**.

Before introducing ABCD matrices, we will first understand physically what happens when a Gaussian beam propagates between mirrors and why a cavity has a particular self-consistent Gaussian mode.

After that, we will introduce ABCD matrices and the Gaussian-beam $q$ parameter.